In [1]:
import sys

sys.path.append("/home/rahim/xelp/work/sqlite-vectordb/")

In [3]:
import pandas as pd
import requests
from src.models import Point, DistanceMetric, CollectionMeta
from src.client import Client
from uuid import uuid4
import numpy as np
from sentence_transformers import SentenceTransformer

In [4]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2").to("cuda")

In [5]:
embeddings1 = model.encode("hello")

In [6]:
embeddings2 = model.encode("/home/rahim/xelp/work/qdrant-client/qdrant_local/books.csv")

In [7]:
diff = embeddings1 - embeddings2
print(np.dot(embeddings1, embeddings2))

0.00031258538


In [8]:
df = pd.read_csv("/home/rahim/xelp/work/qdrant-client/qdrant_local/books.csv")

In [9]:
df.head()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,Unnamed: 12
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.,NaN
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.,NaN
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic,NaN
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.,NaN
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic,NaN


In [10]:
df.drop(labels=["Unnamed: 12"], axis=1, inplace=True)
df.columns

Index(['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13',
       'language_code', '  num_pages', 'ratings_count', 'text_reviews_count',
       'publication_date', 'publisher'],
      dtype='str')

In [6]:
def get_embedding(prompt: str):
    res = requests.post(
        url="http://localhost:8000/v1/embeddings",
        json={
            "model": "sentence-transformers/all-MiniLM-L6-v2",
            "input": "hello",
            "truncate_prompt_tokens": 256,
        },
        timeout=10,
    )
    res.raise_for_status()
    return res.json()["data"][0]["embedding"]

In [7]:
embedding_ = get_embedding("hello")

In [8]:
len(embedding_)

384

In [11]:
client = Client("./test_db")
client.create_collection(
    "books_collection",
    CollectionMeta(
        collection_name="books_collection",
        embedding_size=384,
        distance_metric=DistanceMetric.COSINE,
    ),
)

In [12]:
collection = client.get_collection("books_collection")

In [13]:
columns = df.columns
points = []
for index, row in df.iterrows():
    prompt = ""
    payload = dict()
    for col in columns:
        prompt += col + ": " + str(row[col]) + "\n"
        payload[col] = row[col]
        # embedding = get_embedding(prompt)

    embedding = model.encode(prompt)
    embedding = embedding / np.linalg.norm(embedding)
    point = Point(
        id=str(uuid4()), content=prompt, embedding=embedding.tolist(), payload=payload
    )
    points.append(point)
    if index > 500:
        break

collection.store_points(points=points)

In [14]:
len(collection.ids)

502

In [15]:
query_prompt = "title: Harry Potter\nAuthors: J.K Rowling"
query_embedding = model.encode(query_prompt)

In [16]:
embeddings = np.array(collection.embeddings)
embeddings.shape

(502, 384)

In [19]:
query_embedding = query_embedding / np.linalg.norm(query_embedding)

In [20]:
scores = np.dot(embeddings, query_embedding)

In [21]:
np.dot(embeddings[0], embeddings[4])

np.float64(0.786156011349749)

In [22]:
scores.argmax()

np.int64(6)

In [23]:
collection.ids[6]

'd8a397d3-68c8-4a1d-adae-1f000e132f61'

In [25]:
collection.db.load_point("d8a397d3-68c8-4a1d-adae-1f000e132f61")

Point(id='d8a397d3-68c8-4a1d-adae-1f000e132f61', content='bookID: 10\ntitle: Harry Potter Collection (Harry Potter  #1-6)\nauthors: J.K. Rowling\naverage_rating: 4.73\nisbn: 439827604\nisbn13: 9780439827607\nlanguage_code: eng\n  num_pages: 3342\nratings_count: 28242\ntext_reviews_count: 808\npublication_date: 9/12/2005\npublisher: Scholastic\n', payload={'bookID': 10, 'title': 'Harry Potter Collection (Harry Potter  #1-6)', 'authors': 'J.K. Rowling', 'average_rating': '4.73', 'isbn': '439827604', 'isbn13': 9780439827607, 'language_code': 'eng', '  num_pages': '3342', 'ratings_count': 28242, 'text_reviews_count': 808, 'publication_date': '9/12/2005', 'publisher': 'Scholastic'}, embedding=[-0.04248419031500816, -0.028171909973025322, 0.0024987547658383846, 0.041762858629226685, -0.14663811028003693, 0.04937617853283882, -0.01441285666078329, -0.00431456184014678, 0.014802303165197372, -0.02727227658033371, -0.06952619552612305, 0.011710213497281075, 0.030407432466745377, -0.056607492268

In [20]:
collection.db.close()